# NB07 — Country Export Engine

**Objective**: Generate three CSV variants per country from the validated unified dataset.

## Three output variants

| Variant | Filename | Content |
|---------|----------|---------|
| Observed-only | `{iso3}_soil_observed.csv` | WoSIS measured values at stations with ≥ 1 observed property |
| SoilGrids-only | `{iso3}_soil_soilgrid_only.csv` | SoilGrids 250m raster values (scaled to WoSIS units) at all USABLE stations |
| Unified | `{iso3}_soil_unified.csv` | Observed priority + SoilGrids fallback, source-flagged per property |

## Columns

**Observed-only**: `station_id, lat, lon, iso3, country, QC_FLAG, {property}...`
- Only stations where at least one property was observed
- Only observed-value columns (no `_src` flags, no SoilGrids-only columns)

**SoilGrids-only**: `station_id, lat, lon, iso3, country, QC_FLAG, {sg_property}...`
- All USABLE stations
- SoilGrids columns renamed without `sg_` prefix for clarity
- Scale factors already applied (WR × 100 to match WoSIS %)

**Unified**: `station_id, lat, lon, iso3, country, QC_FLAG, {property}, {property}_src...`
- All USABLE stations
- `_src` column per property: `observed | soilgrids | missing`
- Transparent provenance for every data point

**Inputs**:
- `data/intermediate/unified_surface_all.parquet` (NB05)
- `data/intermediate/soilgrids_surface_all.parquet` (NB04)

**Outputs**:
- `data/output/{iso3}/{iso3}_soil_observed.csv`
- `data/output/{iso3}/{iso3}_soil_soilgrid_only.csv`
- `data/output/{iso3}/{iso3}_soil_unified.csv`
- `reports/nb07_export_summary.md`
- `logs/nb07_country_export.log`

In [1]:
import logging
from pathlib import Path
from datetime import datetime, date

import numpy as np
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
BASE       = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
INTER_DIR  = BASE / 'data' / 'intermediate'
OUTPUT_DIR = BASE / 'data' / 'output'
REPORT_DIR = BASE / 'reports'
LOG_DIR    = BASE / 'logs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'nb07_country_export.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_path, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('nb07')
log.info(f'NB07 started at {datetime.now().isoformat()}')

# ── Column definitions ─────────────────────────────────────────────────────
META_COLS = ['station_id', 'lat', 'lon', 'iso3', 'country', 'QC_FLAG']

# All WoSIS property columns (observed domain)
OBS_PROPS = [
    'BD', 'CEC', 'CF', 'clay', 'N', 'occ', 'pH', 'sand', 'silt',
    'WR_volumetric', 'WR_gravimetric',
    'Al', 'Ca', 'CaCO3', 'Cu', 'EC', 'Fe', 'K', 'Mg', 'Mn', 'Na', 'nematode', 'P', 'TC'
]

# SoilGrids fallback properties and their scale factors (from NB05)
# Used to scale raw sg_* columns to WoSIS units for the SoilGrids-only CSV
SG_FALLBACK = {
    'BD':             ('sg_BD',              1.0),
    'CEC':            ('sg_CEC',             1.0),
    'CF':             ('sg_CF',              1.0),
    'clay':           ('sg_clay',            1.0),
    'N':              ('sg_N',               1.0),
    'occ':            ('sg_occ_carbon',      1.0),
    'pH':             ('sg_pH',              1.0),
    'sand':           ('sg_sand',            1.0),
    'silt':           ('sg_silt',            1.0),
    'WR_volumetric':  ('sg_WR_volumetric',  100.0),
    'WR_gravimetric': ('sg_WR_gravimetric', 100.0),
}

print('NB07 — Country Export Engine')
print(f'Output directory: {OUTPUT_DIR}')

2026-03-19 23:24:04,900 [INFO] NB07 started at 2026-03-19T23:24:04.900427


NB07 — Country Export Engine
Output directory: /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/data/output


In [2]:
# ── Load inputs ────────────────────────────────────────────────────────────
unified = pd.read_parquet(INTER_DIR / 'unified_surface_all.parquet')
sg_raw  = pd.read_parquet(INTER_DIR / 'soilgrids_surface_all.parquet')

log.info(f'Unified loaded   : {len(unified):,} stations × {len(unified.columns)} columns')
log.info(f'SoilGrids loaded : {len(sg_raw):,} stations × {len(sg_raw.columns)} columns')

countries = sorted(unified['iso3'].unique())
log.info(f'Countries to export: {len(countries)}')
print(f'Unified   : {len(unified):,} stations')
print(f'Countries : {len(countries)}')

2026-03-19 23:24:04,981 [INFO] Unified loaded   : 48,435 stations × 56 columns


2026-03-19 23:24:04,982 [INFO] SoilGrids loaded : 48,435 stations × 18 columns


2026-03-19 23:24:04,984 [INFO] Countries to export: 23


Unified   : 48,435 stations
Countries : 23


In [3]:
# ── Build scaled SoilGrids table ──────────────────────────────────────────
# Rename sg_* columns to plain property names and apply scale factors.
# This is the SoilGrids-only export domain: properties where raster data exists.

sg_meta = sg_raw[['station_id', 'lat', 'lon', 'iso3', 'country']].copy()

# Add QC_FLAG from unified (same station set)
qc_map = unified.set_index('station_id')['QC_FLAG']
sg_meta['QC_FLAG'] = sg_meta['station_id'].map(qc_map)

sg_scaled = sg_meta.copy()
for wosis_prop, (sg_col, scale) in SG_FALLBACK.items():
    if sg_col in sg_raw.columns:
        sg_scaled[wosis_prop] = sg_raw[sg_col] * scale

# Also keep extra SoilGrids-only columns (already in WoSIS-compatible naming)
for extra_col in ['sg_occ_ocd', 'sg_WR_volumetric2']:
    if extra_col in unified.columns:
        sg_scaled[extra_col] = unified.set_index('station_id')[extra_col].reindex(
            sg_scaled['station_id'].values
        ).values

log.info(f'SoilGrids scaled table: {len(sg_scaled):,} rows × {len(sg_scaled.columns)} columns')
print(f'SoilGrids scaled table: {sg_scaled.columns.tolist()}')

2026-03-19 23:24:05,187 [INFO] SoilGrids scaled table: 48,435 rows × 19 columns


SoilGrids scaled table: ['station_id', 'lat', 'lon', 'iso3', 'country', 'QC_FLAG', 'BD', 'CEC', 'CF', 'clay', 'N', 'occ', 'pH', 'sand', 'silt', 'WR_volumetric', 'WR_gravimetric', 'sg_occ_ocd', 'sg_WR_volumetric2']


In [4]:
# ── Per-country export loop ────────────────────────────────────────────────
export_summary = []

# Columns for observed-only CSV: meta + all OBS_PROPS present in unified
obs_value_cols = [p for p in OBS_PROPS if p in unified.columns]

# Columns for unified CSV: meta + property values + _src flags
src_cols    = [f'{p}_src' for p in OBS_PROPS if f'{p}_src' in unified.columns]
prop_cols   = [p for p in OBS_PROPS if p in unified.columns]
# Interleave: prop, prop_src, next_prop, next_prop_src ...
unified_value_cols = []
for p in prop_cols:
    unified_value_cols.append(p)
    if f'{p}_src' in unified.columns:
        unified_value_cols.append(f'{p}_src')

# SoilGrids-only columns
sg_prop_cols = list(SG_FALLBACK.keys()) + [c for c in ['sg_occ_ocd', 'sg_WR_volumetric2']
                                            if c in sg_scaled.columns]

for iso3 in countries:
    country_dir = OUTPUT_DIR / iso3
    country_dir.mkdir(exist_ok=True)

    uni_ctry = unified[unified['iso3'] == iso3].copy()
    sg_ctry  = sg_scaled[sg_scaled['iso3'] == iso3].copy()
    n_total  = len(uni_ctry)

    # ── Variant 1: Observed-only ──────────────────────────────────────────
    # Keep only stations with at least one observed property (not all NaN)
    obs_df = uni_ctry[META_COLS + obs_value_cols].copy()
    has_obs = obs_df[obs_value_cols].notna().any(axis=1)
    obs_df  = obs_df[has_obs]
    out_obs = country_dir / f'{iso3}_soil_observed.csv'
    obs_df.to_csv(out_obs, index=False)

    # ── Variant 2: SoilGrids-only ─────────────────────────────────────────
    sg_out_cols = [c for c in META_COLS + sg_prop_cols if c in sg_ctry.columns]
    sg_df = sg_ctry[sg_out_cols].copy()
    out_sg = country_dir / f'{iso3}_soil_soilgrid_only.csv'
    sg_df.to_csv(out_sg, index=False)

    # ── Variant 3: Unified ────────────────────────────────────────────────
    uni_out_cols = [c for c in META_COLS + unified_value_cols if c in uni_ctry.columns]
    uni_df = uni_ctry[uni_out_cols].copy()
    out_uni = country_dir / f'{iso3}_soil_unified.csv'
    uni_df.to_csv(out_uni, index=False)

    # ── Compute summary stats ─────────────────────────────────────────────
    n_obs_stations = has_obs.sum()
    n_sg_stations  = sg_df[list(SG_FALLBACK.keys())].notna().any(axis=1).sum()
    src_flag_cols  = [f'{p}_src' for p in OBS_PROPS if f'{p}_src' in uni_ctry.columns]
    n_obs_cells    = (uni_ctry[src_flag_cols] == 'observed').sum().sum()
    n_sg_cells     = (uni_ctry[src_flag_cols] == 'soilgrids').sum().sum()
    n_miss_cells   = (uni_ctry[src_flag_cols] == 'missing').sum().sum()
    total_cells    = n_total * len(src_flag_cols)

    export_summary.append({
        'iso3': iso3,
        'country': uni_ctry['country'].iloc[0],
        'n_total_stations': n_total,
        'n_observed_stations': n_obs_stations,
        'n_sg_stations': n_sg_stations,
        'pct_obs_cells': round(100 * n_obs_cells / total_cells, 1) if total_cells else 0,
        'pct_sg_cells': round(100 * n_sg_cells / total_cells, 1) if total_cells else 0,
        'pct_missing_cells': round(100 * n_miss_cells / total_cells, 1) if total_cells else 0,
        'obs_csv_kb': round(out_obs.stat().st_size / 1024, 1),
        'sg_csv_kb': round(out_sg.stat().st_size / 1024, 1),
        'unified_csv_kb': round(out_uni.stat().st_size / 1024, 1),
    })

    log.info(f'  {iso3}: {n_total} stations | obs_stations={n_obs_stations} | "\
              obs_cells={n_obs_cells} sg_cells={n_sg_cells} miss_cells={n_miss_cells}')

summary_df = pd.DataFrame(export_summary)
log.info(f'Export loop complete: {len(countries)} countries')
print(f'\nExport complete: {len(countries)} countries')

2026-03-19 23:24:05,632 [INFO]   AGO: 1571 stations | obs_stations=1571 | "              obs_cells=14361 sg_cells=8229 miss_cells=15114


2026-03-19 23:24:05,650 [INFO]   ALB: 67 stations | obs_stations=67 | "              obs_cells=543 sg_cells=284 miss_cells=781


2026-03-19 23:24:07,504 [INFO]   AUS: 33748 stations | obs_stations=33748 | "              obs_cells=137470 sg_cells=253872 miss_cells=418610


2026-03-19 23:24:07,527 [INFO]   BDI: 28 stations | obs_stations=28 | "              obs_cells=241 sg_cells=88 miss_cells=343


2026-03-19 23:24:07,577 [INFO]   BEN: 716 stations | obs_stations=716 | "              obs_cells=5611 sg_cells=2810 miss_cells=8763


...

2026-03-19 23:24:08,486 [INFO]   PNG: 16 stations | obs_stations=16 | "              obs_cells=132 sg_cells=71 miss_cells=181


2026-03-19 23:24:08,500 [INFO]   TCD: 13 stations | obs_stations=13 | "              obs_cells=104 sg_cells=57 miss_cells=151


2026-03-19 23:24:08,521 [INFO]   TUN: 40 stations | obs_stations=40 | "              obs_cells=347 sg_cells=126 miss_cells=487


2026-03-19 23:24:08,523 [INFO] Export loop complete: 23 countries



Export complete: 23 countries


In [5]:
# ── Print summary table ────────────────────────────────────────────────────
print('=== NB07 EXPORT SUMMARY ===')
print(f'{"ISO3":5s}  {"Stations":>9}  {"Obs stations":>13}  {"Obs%":>6}  {"SG%":>6}  {"Miss%":>7}  '
      f'{"obs_kb":>8}  {"sg_kb":>7}  {"uni_kb":>7}')
print('-' * 90)
for _, row in summary_df.iterrows():
    print(f'{row["iso3"]:5s}  {row["n_total_stations"]:9,}  {row["n_observed_stations"]:13,}  '
          f'{row["pct_obs_cells"]:5.1f}%  {row["pct_sg_cells"]:5.1f}%  {row["pct_missing_cells"]:6.1f}%  '
          f'{row["obs_csv_kb"]:8.1f}  {row["sg_csv_kb"]:7.1f}  {row["unified_csv_kb"]:7.1f}')

total_files = len(countries) * 3
total_kb    = summary_df[['obs_csv_kb', 'sg_csv_kb', 'unified_csv_kb']].sum().sum()
print(f'\nTotal: {total_files} files  |  {total_kb / 1024:.1f} MB  |  {OUTPUT_DIR}')

log.info(f'NB07 summary: {total_files} files, {total_kb:.0f} KB total')

2026-03-19 23:24:08,536 [INFO] NB07 summary: 69 files, 29477 KB total


=== NB07 EXPORT SUMMARY ===
ISO3    Stations   Obs stations    Obs%     SG%    Miss%    obs_kb    sg_kb   uni_kb
------------------------------------------------------------------------------------------
AGO        1,571          1,571   38.1%   21.8%    40.1%     260.0    179.1    584.8
ALB           67             67   33.8%   17.7%    48.6%       9.6      8.3     23.4
AUS       33,748         33,748   17.0%   31.3%    51.7%    4634.4   4129.1  11592.4
BDI           28             28   35.9%   13.1%    51.0%       4.1      3.2      9.9
BEN          716            716   32.7%   16.4%    51.0%      90.0     77.7    235.4
BFA        1,671          1,671   22.8%   26.7%    50.5%     239.2    216.6    582.6
BWA        1,253          1,253   35.4%   24.5%    40.1%     186.2    143.9    446.2
CAF           80             80   34.2%   15.9%    49.9%      13.4     10.2     29.8
CMR        1,323          1,323   29.5%   21.4%    49.1%     203.2    165.8    473.9
DEU        2,873          2,873

In [6]:
# ── Write export report ────────────────────────────────────────────────────
summary_md = summary_df.to_markdown(index=False)

report = f"""# NB07 — Country Export Report

**Date**: {date.today().isoformat()}
**Notebook**: `notebook/NB07_country_export.ipynb`
**Status**: COMPLETED — no errors

---

## 1. Output Structure

```
data/output/
  {{iso3}}/
    {{iso3}}_soil_observed.csv      ← WoSIS observations only
    {{iso3}}_soil_soilgrid_only.csv ← SoilGrids raster values (scaled to WoSIS units)
    {{iso3}}_soil_unified.csv       ← Observed priority + SoilGrids fallback + _src flags
```

Total: **{total_files} files** across {len(countries)} countries  |  **{total_kb / 1024:.1f} MB**

---

## 2. Per-Country Export Summary

- **Obs%** = % of (station × property) cells filled from WoSIS observations
- **SG%** = % of cells filled from SoilGrids fallback
- **Miss%** = % of cells still NaN (neither observed nor raster available)

{summary_md}

---

## 3. Column Dictionary

### Observed-only CSV
| Column | Description | Unit |
|--------|-------------|------|
| station_id | Unique station identifier | — |
| lat / lon | WGS84 coordinates | degrees |
| iso3 | ISO 3166-1 alpha-3 country code | — |
| country | Country name | — |
| QC_FLAG | Coordinate quality flag (VALID, NEAR_BORDER, OVERSEAS_TERRITORY) | — |
| BD | Bulk density (observed) | g/cm³ |
| CEC | Cation exchange capacity | cmol(c)/kg |
| CF | Coarse fragments | % |
| clay / sand / silt | Particle size fractions | g/kg |
| N | Total nitrogen | g/kg |
| occ | Organic carbon content | g/kg |
| pH | Soil pH (H2O) | — |
| WR_volumetric | Volumetric water retention (0.01 bar) | % |
| WR_gravimetric | Gravimetric water retention (15 bar) | % |
| EC | Electrical conductivity | dS/m |
| CaCO3 | Calcium carbonate | g/kg |
| P | Available phosphorus | mg/kg |
| TC | Total carbon | g/kg |
| Al, Ca, Cu, Fe, K, Mg, Mn, Na | Extractable elements | cmol(c)/kg or mg/kg |
| nematode | Nematode count | count/g |

### SoilGrids-only CSV
Same metadata columns. Property columns are SoilGrids 250m values **scaled to WoSIS units** (WR × 100).
Extra: `sg_occ_ocd` (organic carbon density, g/dm³), `sg_WR_volumetric2` (0.33 bar, cm³/cm³).

### Unified CSV
Same property columns as observed. Each property has a companion `{{prop}}_src` column:
- `observed` — value from WoSIS measurement
- `soilgrids` — value from SoilGrids 250m raster (scaled)
- `missing` — no data available

---

## 4. Pipeline Complete

```
NB01 — Data Audit & Country Mapping      ✓
NB02 — Surface Extraction (WoSIS → wide) ✓  53,708 stations × 29 properties
NB03 — Coordinate Validation & QC        ✓  48,435 USABLE (90.2%)
NB04 — SoilGrids Raster Extraction       ✓  48,435 stations × 13 SG layers
NB05 — Integration (observed + fallback) ✓  WR unit fix (×100), source flags
NB06 — Validation                        ✓  All STOP checks passed
NB07 — Country Export                    ✓  {total_files} CSV files
```
"""

report_path = REPORT_DIR / 'nb07_export_report.md'
report_path.write_text(report)

log.info(f'Export report written: {report_path}')
log.info('NB07 completed successfully — pipeline DONE')
print(f'\nReport saved: {report_path}')
print('\nPipeline complete.')

2026-03-19 23:24:08,700 [INFO] Export report written: /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/reports/nb07_export_report.md


2026-03-19 23:24:08,701 [INFO] NB07 completed successfully — pipeline DONE



Report saved: /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/reports/nb07_export_report.md

Pipeline complete.
